<a href="https://colab.research.google.com/github/Rimix-oups/milestones/blob/main/Milestones_personnel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Group project — SeqTools: basic processing of biological sequences
**Python for Bioinformatics · MSc 1**

**Group name:** ...

| Member | Main responsibilities |
|---|---|
| ... | ... |
| ... | ... |
| ... | ... |

**How to work together**
1. ONE member: `File → Save a copy in Drive`, rename it `GroupX_seqtools.ipynb`, then `Share` → add the other members
   as **Editors**.
2. Everyone works in this shared notebook, in the section of the current milestone. Colab saves automatically,
   but two people should not edit **the same cell** at the same time.
3. Keep the **contribution log** at the end up to date: it is used for the individual part of the grade.

The full instructions are in the project brief (`Project_SeqTools.pdf`).

## ⚙️ Setup — run these cells every time you open the notebook
The Colab disk is erased when the session ends: these cells re-create the data file.

In [ ]:
%%writefile sequences.fasta
>gene_alpha synthetic gene, one ORF on the forward strand
AGGTAAGGTGGTTGAGATCTGGACTTTTGACGCCTGGAGCATGCCGCAGTGCTCCTCGAA
AAGTAGCCATGCCTTGGGCTGCTCCCAGGCGCTCCGTTGTTCTAGTGCTTTCACCAGCCA
AGTATTGGACGTTCGAAGTGTTTCCAATACCCCCATGCCGGACATAGTCTGTATCCCACC
CCTCCCCATTTTATTGCTTGTCAGGGAGGGACCGACTAGTGAGGCTCCCACTTCAAAAGG
TCTGGGTCACTTGTGTCTCGCAGCTTATCTGCGGAGAACGGAGTAGCCTGTAAGGGTTAA
GCTAGAAATTTCGGATGAAAGAGTGCTGCTCCCTTG
>gene_beta synthetic gene, ORF on the reverse strand
CACCACTGCAACGCGATGTGCACGTTGTAATTAACACAACCCTGCGCAACAACCCCAACG
CCGGAGGTCGGGGGGCGAGCCTATTCCGTCCGAATACTCGCCCCGACCCTGAAACCTCGT
GTCGCCACCGCTGGCGCCGCTGGCGAGTGGCCGGCAGCAAGTGGCAGCGTCGCAATGAAG
GCGGTGGTATGTGCCGTACCCTTTGTCCACCGACACAGTGGATCCGCGCATAACTAATTG
AGTCGGGAGGAATTGTTCCTATGAAG
>gene_gamma synthetic gene, lower case, two ORFs
tgcgttcgtagctagcactgtcagtatgagaccgtgccacgtgctaatcggaacttcgat
gcgccccctaaaccgaggttacgttttcaagcatttggacacattggtcatagcaatcaa
aaaccgcaggtctcttgctaggtggctttctaataccctagggtaatgattgtgttttgt
aagcagatgatagggggcacgcaacagaaggccgaacgattttcggccaactccggcggg
acacgggcgtcgggactgggaagcccccagagatcacctgtcgcccagaaggtctgagaa
cggcgcgtgctagagaacgcccacagc
>gene_delta contains ambiguous bases (N)
CCTGGTTCGTGTCCANNNTTTGCTTAAAGCTCAAAAAAGGATGTGCCTAACAGAATTGCG
NCACTATTCTAGCTCACGTCCAGGCCATGTCTTGCTCGGGCTTCGGACAGGCGTATGCCG
TCTATGCCTGGCCCCACTCGGGCTCCCCCCGTCTTACCCTGAATAAAACTGCGGGAATAG
AATTTG
>gene_epsilon nested start codons
TTCTTTGGCAGAATACGCAGATGACTGTTCCGCCTGGGATCCGAATAAATATGATCTACC
TTAGAGACCGTAAGTTAACACGCAACAGATTTAGTACGACCCTTACCGCCCATAGACATA
CGCCGACGGTCCGCCGCGCCTACGGAAGCGGCAACGCATCATTAAGATCTGACTTGGGGC
TTATGTGTGACATTGCCTTGTGGTAGCGTGCCCGGGTATCATGCGAGATTA
>gene_zeta short fragment, no ORF
CCGTTAGCATTAGC
>gene_eta typing error in the sequence
CAAAATCATGCTACACTCGGACGTTCAAAAGTACGAAAACACATAGTATCATGCCX1AGT
TTCACCGGGATATACTTGACCTAGTGTTTGCCCTACT
>gene_theta empty record
>gene_iota GC-rich synthetic gene
GACGGTGGCGACCTGGCCGGATGCGGCAGGTAGGAGCCGACCCTGGGGAATATGGAAATA
ACGCCGTCTTCTGTCAGATCAAGCAACCCGGGCCACCCCATGAGTACTCAGTCTCGACCG
GCGGCCCATTACCCCCGAGAGGGTCGTACGGACGGTACCGGGCCTGCCCACCGGCGCGCG
AGGTGCCGTAGCCTGAGCACAGCCGACGCGT

Writing sequences.fasta


In [ ]:
import os
os.makedirs("data", exist_ok=True)
os.replace("sequences.fasta", "data/sequences.fasta")
print(os.listdir("data"))

['sequences.fasta']


In [ ]:
# Standard genetic code (NCBI table 1). '*' = stop codon
GENETIC_CODE = {
    'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
    'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
    'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
    'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
    'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
    'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
    'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
    'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
    'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
    'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
    'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
    'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
    'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
    'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
    'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
    'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G',
}

TEST_SEQ = "ATGTGCTCATGCCTCTTCACATACTTCCGCAGGCGATACTAA"   # small test sequence for the first milestones
print(len(GENETIC_CODE), "codons")

64 codons


---
## Milestone 1 — Get organised and first computations (session 1)

1. Form **groups of 3–4**. One person opens `00_Project_Starter.ipynb`, saves a copy in Drive
   and **shares it** with the group (Share → Editor). Rename it `GroupX_seqtools.ipynb`.
2. Read the project brief together (`Project_SeqTools.pdf`), and split the work.
3. In the starter notebook, section *Milestone 1*: store the test sequence in a variable and use
   **variables and operators only** to compute:
   - the length of the sequence
   - the number of A, C, G and T (you may use `seq.count("A")`)
   - the GC content in %
   - whether it starts with `"ATG"` (`in`, or slicing, which we will see in session 4)

In [ ]:

seq = TEST_SEQ

seq_length = len(seq)



n_A = seq.count("A")
n_C = seq.count("C")
n_T = seq.count("T")
n_G = seq.count("G")

gc = ((n_C + n_G)/seq_length) * 100

if seq[:3] == "ATG":
    print("Sequence starts with 'ATG'")
else:
    print("Sequence doesn't start with 'ATG'")

print("Number of A:", n_A, "Number of C:", n_C, "Number of T:", n_T, "Number of G:", n_G)
print("The sequence length is:", seq_length, "nt")
print("GC content is:", round(gc), "%")


Sequence starts with 'ATG'
Number of A: 10 Number of C: 13 Number of T: 12 Number of G: 7
The sequence length is: 42 nt
GC content is: 48 %


---
## Milestone 2 — Validation with loops and conditions (session 2)

In the group notebook, section *Milestone 2*:
1. **Task 1 (logic only):** loop over the characters of a sequence and decide if it is a valid DNA
   sequence (only A, C, G, T, upper or lower case). Print `True`/`False`. Test it on `"atggagataac"` and
   `"ATGCXAGA"`.
2. Classify the GC content as low / medium / high with `if/elif/else`.
3. Use `range(0, len(seq), 3)` to print the sequence codon by codon.

In [ ]:
seq = input("Enter a valid DNA sequence: ").upper()

if len(seq)==0:     #empêche de calculer une sequence vide
  print("DNA sequence too short.")
else:      #si sequence pas vide, effectue les vérifications
  for b in seq:
    if b == "A":
      continue
    if b == "C":
      continue
    if b == "T":
      continue
    if b == "G":
      continue
    else:
      print("DNA sequence is not valid.")
      break

  else:
    gc = ((seq.count("G") + seq.count("C"))/len(seq)) * 100

    if gc < 40:
      print ("Low GC content")
    elif 40 <= gc <= 60:
      print ("Medium GC content")
    else:
      print ("High GC content")

    for i in range(0, len(seq),3):
        codons = seq[i:i+3]
        if len(codons) == 3:
          print("Position", i, codons)


Enter a valid DNA sequence: xxatg
DNA sequence is not valid.


---
## Milestone 3 — Dictionaries for frequencies and the genetic code (session 3)

In the group notebook, section *Milestone 3*:
1. **Task 2:** build a dictionary with the frequency of each symbol in a sequence.
2. Load the `GENETIC_CODE` dictionary (in the starter) and translate the test sequence codon by codon.
3. **Task 3 (draft):** count the amino acids of a protein sequence and print them from most to least frequent
   (`sorted(..., key=..., reverse=True)`).
4. **Task 9 (draft):** for one amino acid (e.g. `'L'`), count how often each of its codons is used.

In [ ]:
seq = TEST_SEQ

f_s = {"A": 0, "C": 0, "T": 0, "G": 0}
other_symbols = {}     #dictionaire vide pour les symboles qui ne sont pas des nt

for nt in seq:
  if nt in f_s: #verifie si le nucleotide examiné par b est dans le dictionaire ou pas
    f_s[nt] += 1
  else:
    if nt in other_symbols: #complète le dictionaire avec des symboles différent de ACTG
      other_symbols[nt] += 1
    else:
      other_symbols[nt] = 1
      print("symbol not valid in dna sequence")

print("frequencies of nucleotides:",f_s)
print("Other symbols found:", other_symbols)


list_aa=[]
for i in range(0, len(seq) - 2, 3):  # -2 given by AI: # Le -2 évite de commencer un codon s'il reste moins de 3 nucléotides à lire.
    codon = seq[i:i+3]
    aa = GENETIC_CODE[codon]
    list_aa.append(aa)

seq_aa = "".join(list_aa)
print(seq_aa)

freq_aa = {}

for aa in seq_aa:
  if aa in freq_aa:
    freq_aa[aa] += 1
  else:
    freq_aa[aa] = 1

sorted(freq_aa.items(), key=lambda x: x[1], reverse=True)

freq_codons = {}                                     #pour optimiser peut être modifer le code avec un input pour pas se limiter à "L"
for nt in range(0, len(seq)-2, 3):
  codons = seq[nt:nt+3]
  if GENETIC_CODE[codons] == "L":
    if codons in freq_codons:
      freq_codons[codons] +=1
    else:
      freq_codons[codons] = 1

print(freq_codons)


frequencies of nucleotides: {'A': 10, 'C': 13, 'T': 12, 'G': 7}
Other symbols found: {}
MCSCLFTYFRRRY*
{'CTC': 1}


---
## Milestone 4 — String processing: transcription, reverse complement, frames (session 4)

In the group notebook, section *Milestone 4*:
1. **Task 4:** GC content with string methods.
2. **Task 5:** transcription (`replace`).
3. **Task 6:** reverse complement.
4. **Tasks 7–8:** translate from a given start position.
5. **Task 10 (draft):** build the 6 reading frames (3 forward and 3 on the reverse complement).

In [ ]:
# YOUR CODE HERE

---
## Milestone 5 — Refactor everything into functions (session 5)

In the group notebook, section *Milestone 5*:
1. Turn every piece of code you wrote into a **function with a docstring** (purpose, inputs, output).
   Use the names from the brief (`validate_dna`, `frequency`, `gc_content`, `transcription`, ...).
2. `transcription` must **raise a `ValueError`** if the DNA is not valid.
3. **Tasks 11–14:** `all_proteins`, `has_stop_codon`, `validate_protein`, `all_orfs`.
4. Write at least one `assert` test per function (e.g. `assert validate_dna("ATGC") == True`).

In [ ]:
# YOUR CODE HERE

---
## Milestone 6 — Module + main program (session 6)

In the group notebook, section *Milestone 6*:
1. Move all your functions into **`seqtools.py`** with `%%writefile seqtools.py`.
2. Write **`main.py`**: it does `import seqtools`, asks the user for a DNA sequence with `input()`,
   checks it and prints a small report (use `datetime` to add the date and time of the analysis).
3. Run it from the notebook: `!python main.py`.
4. Tip: put your module under a `if __name__ == "__main__":` guard if it has test code.

In [ ]:
%%writefile seqtools.py
"""seqtools — basic processing of biological sequences (DNA, RNA, proteins).

Group: ...
"""

GENETIC_CODE = {}   # paste the dictionary here


def validate_dna(seq):
    """Task 1 — Check that seq is a valid DNA sequence.

    Input: seq (str), a DNA sequence, upper or lower case.
    Output: True if seq only contains A, C, G, T, False otherwise.
    """
    ...

# ... paste all your other functions here, each one with a docstring ...

In [ ]:
%%writefile main.py
"""main.py — SeqTools main program. Group: ..."""
import seqtools

dna = input("Enter a DNA sequence: ")
# check the sequence, then apply your functions and print a report

In [ ]:
!python main.py

---
## Milestone 7 — Working with FASTA files (session 7)

In the group notebook, section *Milestone 7*:
1. **Tasks 15–16:** `read_fasta(path)` → dictionary `{id: sequence}` and `write_fasta(records, path)`.
2. **Task 17:** `write_summary(...)`: one CSV line per sequence (id, length, GC%, valid, number of ORFs,
   length of the longest protein).
3. Update `main.py` so it takes the FASTA file from the command line:
   `!python main.py data/sequences.fasta` (`sys.argv`), and writes its outputs in `results/`.

In [ ]:
# Test your file functions here, then update seqtools.py and main.py

In [ ]:
!python main.py data/sequences.fasta
!ls results

---
## Milestone 8 — Robustness, final polish and oral presentation (session 8)

1. **Task 18:** define `InvalidSequenceError` and raise it from your functions. In `main.py`, catch errors
   (missing file, invalid records) so that **one bad sequence never stops the whole run**. Write the
   problems to `results/run.log` with a timestamp.
2. Run the complete pipeline on `data/sequences.fasta` (it contains some traps!).
3. Prepare your **oral presentation** (10 min + 5 min questions): live demo in Colab, design choices,
   one difficulty you solved, who did what.

In [ ]:
# Final run: this is what you will show during the oral presentation
!python main.py data/sequences.fasta
!cat results/run.log
!head results/summary.csv

---
## 📝 Contribution log
| Session | Member | What I did |
|---|---|---|
| 1 | | |
| 2 | | |